# Stromal/Vascular Reintegration v1.3
**Author:** r2end | **Date:** 2026-03-08

## Hierarchy redesign (v1.2 → v1.3)
- L1 = `Stromal` (entire lineage)
- L2 = broad category: `Endothelial` / `Fibroblast` / `Pericyte` / `Smooth Muscle` / `Schwann`
- L3 = original cluster prefix without `_c{N}` suffix (e.g. `Endothelia_vascular_arterial_pulmonary`)
- scANVI training label now uses L3 (all subclusters of same type share one label)
- ANNOTATION_TABLE restructured: columns = [cluster, L2, L3, action]
- Step 3 (manual reassign) replaced by table-driven L2/L3 assignment
- PULMONARY_ALVEOLAR filter derived from L3 values

## Prior fixes retained from v1.1
- [1] Critical Step11: removed redundant SCANVI.setup_anndata
- [2] Perf Step10: vectorized kNN purity
- [3] Memory Step11: del model_scvi before scANVI train
- [4] Minor Step7: counts fallback from .raw not log1p .X


## Imports + Environment + Configuration + Annotation Table

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Stromal/Vascular Reintegration: Contamination Removal -> scVI -> Tissue-aware scANVI
======================================================================================
Author: r2end  |  Date: 2026-03-08  |  Version: v1.3

Hierarchy (v1.3 redesign):
  L1 = Stromal                         (entire lineage)
  L2 = Endothelial / Fibroblast / Pericyte / Smooth_Muscle / Schwann
  L3 = cluster prefix without _c{N}   (e.g. Endothelia_vascular_arterial_pulmonary)

scANVI training label = L3
  All subclusters within the same L3 group share one semi-supervised label.
  Tissue-aware Unknown assignment and kNN purity filter are applied at the L3 level.

DROP clusters (contamination):
  Endothelia_vascular_Cap_a_c0          -> Interferon-activated Immune Contaminants
  Endothelia_vascular_venous_systemic_c3 -> MHC-II-high APC-like Contaminants
  Endothelia_vascular_venous_systemic_c5 -> Plasma/B-cell Contaminants
  Fibro_myofibroblast_c0                -> Osteogenic Contaminants

REASSIGN clusters (L2/L3 overrides in table):
  Endothelia_vascular_venous_systemic_c4 -> L2=Fibroblast,  L3=Fibro_stress_activated
  Fibro_adventitial_c2                   -> L2=Fibroblast,  L3=Fibro_alveolar
  Muscle_perivascular_immune_recruiting_c1 -> L2=Pericyte,  L3=Muscle_pericyte_systemic
"""

import os
for _k in ["OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"]:
    os.environ[_k] = "8"
import matplotlib; matplotlib.use('Agg')
import warnings; warnings.filterwarnings('ignore')
import gc, re, time, numpy as np, pandas as pd
import importlib
import sympy

try:
    if not hasattr(sympy, 'printing'):
        sympy.printing = importlib.import_module('sympy.printing')
except Exception as e:
    raise RuntimeError(
        f"Sympy import is broken (sympy module path: {getattr(sympy, '__file__', 'unknown')}). "
        "Please ensure real sympy package is available and no local sympy.py shadows it."
    ) from e

import scanpy as sc
import scvi
import torch
import matplotlib.pyplot as plt
from scipy import sparse
from pathlib import Path
from sklearn.neighbors import NearestNeighbors

PIPELINE_START = time.time()

# ============================================================================
# CONFIGURATION  --  UPDATE TISSUE_KEY / LUNG_TRACHEA_VALUES BEFORE RUNNING
# ============================================================================

INPUT_H5AD  = Path("/home/h2048/data/py/0120/stromal_analysis_unified/results/"
                   "subcluster_unified_v2_20260128/"
                   "adata_stromal_subclustered_FINAL_v2_20260128.h5ad")
OUTPUT_DIR  = Path("/home/h2048/data/py/0308/stromal_reintegration_v1_3")
FIG_DIR     = OUTPUT_DIR / "figures"
MODEL_DIR   = OUTPUT_DIR / "models"
OUTPUT_H5AD = OUTPUT_DIR / "stromal_reintegrated_scvi_scanvi_v1_3.h5ad"
for d in [OUTPUT_DIR, FIG_DIR, MODEL_DIR]: d.mkdir(parents=True, exist_ok=True)

# Source L2/L3 columns from original object (will be overwritten by table-driven assignment)
SRC_L2_COL  = 'cell_type_L2'
SRC_L3_COL  = 'cell_type_L3'   # fine subcluster IDs, e.g. Endothelia_vascular_arterial_pulmonary_c0
BATCH_KEY   = 'sample'

# Output obs columns for the new hierarchy
OUT_L1      = 'cell_type_L1'
OUT_L2      = 'cell_type_L2'
OUT_L3      = 'cell_type_L3'

TISSUE_KEY  = 'tissue'
LUNG_TRACHEA_VALUES = {'lung','trachea','Lung','Trachea',
                       'Lung tissue','Tracheal tissue',
                       'lung_tissue','trachea_tissue'}

N_HVG=4000; N_LATENT=75; N_HIDDEN=128; N_LAYERS=2; DROPOUT=0.1
MAX_EPOCHS_SCVI=400; MAX_EPOCHS_SCANVI=200; BATCH_SIZE=256
RANDOM_SEED=42; UNLABELED='Unknown'; MIN_CELLS_BATCH=3
PURITY_K=30; PURITY_THRESHOLD=0.5

# ============================================================================
# REPRODUCIBILITY
# ============================================================================
np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_SEED)
scvi.settings.seed=RANDOM_SEED; scvi.settings.dl_num_workers=0
USE_GPU = torch.cuda.is_available()
sc.settings.verbosity=2; sc.settings.n_jobs=16

# ============================================================================
# ANNOTATION TABLE  (v1.3 redesign)
# ============================================================================
# Columns:
#   cluster  : original fine subcluster ID (e.g. Endothelia_vascular_arterial_pulmonary_c0)
#   L2       : broad cell class (Endothelial / Fibroblast / Pericyte / Smooth_Muscle / Schwann)
#   L3       : cluster prefix without _c{N} suffix
#              = re.sub(r'_c\d+$', '', cluster)  in most cases
#              overridden for REASSIGN clusters to place them in correct L3 group
#   action   : KEEP | REVIEW | DROP | REASSIGN
#              KEEP / REVIEW : retained with assigned L2/L3
#              DROP          : removed (contamination); L2/L3 = 'Contaminants'
#              REASSIGN      : retained with overridden L2/L3 (different from original prefix)
# ============================================================================

ANNOTATION_TABLE = [
    # --- Lymphatic Endothelia ---
    ("Endothelia_Lymphatic_c0",                          "Endothelial",   "Endothelia_Lymphatic",                        "KEEP"),
    ("Endothelia_Lymphatic_c1",                          "Endothelial",   "Endothelia_Lymphatic",                        "REVIEW"),
    ("Endothelia_Lymphatic_c2",                          "Endothelial",   "Endothelia_Lymphatic",                        "KEEP"),
    # --- Capillary aCap ---
    ("Endothelia_vascular_Cap_a_c0",                     "Contaminants",  "Contaminants",                                "DROP"),   # Interferon-activated Immune Contaminants
    ("Endothelia_vascular_Cap_a_c1",                     "Endothelial",   "Endothelia_vascular_Cap_a",                   "KEEP"),
    ("Endothelia_vascular_Cap_a_c2",                     "Endothelial",   "Endothelia_vascular_Cap_a",                   "KEEP"),
    # --- Capillary gCap ---
    ("Endothelia_vascular_Cap_g_c0",                     "Endothelial",   "Endothelia_vascular_Cap_g",                   "REVIEW"),
    ("Endothelia_vascular_Cap_g_c1",                     "Endothelial",   "Endothelia_vascular_Cap_g",                   "KEEP"),
    ("Endothelia_vascular_Cap_g_c2",                     "Endothelial",   "Endothelia_vascular_Cap_g",                   "REVIEW"),
    ("Endothelia_vascular_Cap_g_c3",                     "Endothelial",   "Endothelia_vascular_Cap_g",                   "REVIEW"),
    # --- Arterial pulmonary ---
    ("Endothelia_vascular_arterial_pulmonary_c0",        "Endothelial",   "Endothelia_vascular_arterial_pulmonary",      "KEEP"),
    ("Endothelia_vascular_arterial_pulmonary_c1",        "Endothelial",   "Endothelia_vascular_arterial_pulmonary",      "KEEP"),
    ("Endothelia_vascular_arterial_pulmonary_c2",        "Endothelial",   "Endothelia_vascular_arterial_pulmonary",      "REVIEW"),
    # --- Arterial systemic ---
    ("Endothelia_vascular_arterial_systemic_c0",         "Endothelial",   "Endothelia_vascular_arterial_systemic",       "KEEP"),
    ("Endothelia_vascular_arterial_systemic_c1",         "Endothelial",   "Endothelia_vascular_arterial_systemic",       "REVIEW"),
    # --- Venous pulmonary ---
    ("Endothelia_vascular_venous_pulmonary_c0",          "Endothelial",   "Endothelia_vascular_venous_pulmonary",        "KEEP"),
    ("Endothelia_vascular_venous_pulmonary_c1",          "Endothelial",   "Endothelia_vascular_venous_pulmonary",        "REVIEW"),
    ("Endothelia_vascular_venous_pulmonary_c2",          "Endothelial",   "Endothelia_vascular_venous_pulmonary",        "REVIEW"),
    # --- Venous systemic ---
    ("Endothelia_vascular_venous_systemic_c0",           "Endothelial",   "Endothelia_vascular_venous_systemic",         "REVIEW"),
    ("Endothelia_vascular_venous_systemic_c1",           "Endothelial",   "Endothelia_vascular_venous_systemic",         "KEEP"),
    ("Endothelia_vascular_venous_systemic_c2",           "Endothelial",   "Endothelia_vascular_venous_systemic",         "KEEP"),
    ("Endothelia_vascular_venous_systemic_c3",           "Contaminants",  "Contaminants",                                "DROP"),   # MHC-II-high APC-like Contaminants
    ("Endothelia_vascular_venous_systemic_c4",           "Fibroblast",    "Fibro_stress_activated",                      "REASSIGN"), # Stress-activated Fibroblasts (misclassified)
    ("Endothelia_vascular_venous_systemic_c5",           "Contaminants",  "Contaminants",                                "DROP"),   # Plasma/B-cell Contaminants
    # --- Fibroblast adventitial ---
    ("Fibro_adventitial_c0",                             "Fibroblast",    "Fibro_adventitial",                           "KEEP"),
    ("Fibro_adventitial_c1",                             "Fibroblast",    "Fibro_adventitial",                           "KEEP"),
    ("Fibro_adventitial_c2",                             "Fibroblast",    "Fibro_alveolar",                              "REASSIGN"), # Homeostatic Alveolar Fibroblasts (misclassified)
    # --- Fibroblast alveolar ---
    ("Fibro_alveolar_c0",                                "Fibroblast",    "Fibro_alveolar",                              "REVIEW"),
    ("Fibro_alveolar_c1",                                "Fibroblast",    "Fibro_alveolar",                              "KEEP"),
    ("Fibro_alveolar_c2",                                "Fibroblast",    "Fibro_alveolar",                              "KEEP"),
    # --- Fibroblast myofibroblast ---
    ("Fibro_myofibroblast_c0",                           "Contaminants",  "Contaminants",                                "DROP"),   # Osteogenic Contaminants
    ("Fibro_myofibroblast_c1",                           "Fibroblast",    "Fibro_myofibroblast",                         "REVIEW"), # Unresolved Myofibroblasts -- KEPT
    # --- Fibroblast peribronchial ---
    ("Fibro_peribronchial_c0",                           "Fibroblast",    "Fibro_peribronchial",                         "KEEP"),
    ("Fibro_peribronchial_c1",                           "Fibroblast",    "Fibro_peribronchial",                         "KEEP"),
    ("Fibro_peribronchial_c2",                           "Fibroblast",    "Fibro_peribronchial",                         "REVIEW"), # Neural-like stromal; grouped under peribronchial pending resolution
    # --- Pericyte pulmonary ---
    ("Muscle_pericyte_pulmonary_c0",                     "Pericyte",      "Muscle_pericyte_pulmonary",                   "KEEP"),
    ("Muscle_pericyte_pulmonary_c1",                     "Pericyte",      "Muscle_pericyte_pulmonary",                   "REVIEW"),
    ("Muscle_pericyte_pulmonary_c2",                     "Pericyte",      "Muscle_pericyte_pulmonary",                   "KEEP"),
    ("Muscle_pericyte_pulmonary_c3",                     "Pericyte",      "Muscle_pericyte_pulmonary",                   "KEEP"),
    # --- Pericyte systemic ---
    ("Muscle_pericyte_systemic_c0",                      "Pericyte",      "Muscle_pericyte_systemic",                    "KEEP"),
    ("Muscle_pericyte_systemic_c1",                      "Pericyte",      "Muscle_pericyte_systemic",                    "KEEP"),
    # --- Perivascular immune-recruiting ---
    ("Muscle_perivascular_immune_recruiting_c0",         "Smooth_Muscle", "Muscle_perivascular_immune_recruiting",       "KEEP"),
    ("Muscle_perivascular_immune_recruiting_c1",         "Pericyte",      "Muscle_pericyte_systemic",                    "REASSIGN"), # Quiescent pericyte-like; merged with systemic pericytes
    # --- Smooth muscle ---
    ("Muscle_smooth_arterial_systemic_c0",               "Smooth_Muscle", "Muscle_smooth_arterial_systemic",             "KEEP"),
    ("Muscle_smooth_arterial_systemic_c1",               "Smooth_Muscle", "Muscle_smooth_arterial_systemic",             "KEEP"),
    ("Muscle_smooth_pulmonary_c0",                       "Smooth_Muscle", "Muscle_smooth_pulmonary",                     "REVIEW"),
    ("Muscle_smooth_pulmonary_c1",                       "Smooth_Muscle", "Muscle_smooth_pulmonary",                     "KEEP"),
    # --- Schwann ---
    ("Schwann_nonmyelinating_c0",                        "Schwann",       "Schwann_nonmyelinating",                      "KEEP"),
]

annot_df = pd.DataFrame(
    ANNOTATION_TABLE,
    columns=['cluster', 'L2', 'L3', 'action']
).set_index('cluster')

DROP_CLUSTERS     = annot_df[annot_df['action'] == 'DROP'].index.tolist()
REASSIGN_CLUSTERS = annot_df[annot_df['action'] == 'REASSIGN'].index.tolist()

# Derive pulmonary/alveolar L3 labels for tissue-aware Unknown assignment.
# All cells whose L3 label contains 'pulmonary' or 'alveolar' but originate
# from non-lung/trachea tissue will be set to Unknown before scANVI training.
PULMONARY_ALVEOLAR_L3 = sorted(
    annot_df[annot_df['L3'].str.contains('pulmonary|alveolar', case=False, regex=True)]['L3']
    .unique().tolist()
)

print(f"\nDROP clusters ({len(DROP_CLUSTERS)}):")
for c in DROP_CLUSTERS: print(f"  {c}")

print(f"\nREASSIGN clusters ({len(REASSIGN_CLUSTERS)}):")
for c in REASSIGN_CLUSTERS:
    print(f"  {c} -> L2={annot_df.loc[c,'L2']!r}, L3={annot_df.loc[c,'L3']!r}")

print(f"\nPulmonary/Alveolar L3 labels for tissue-aware scANVI ({len(PULMONARY_ALVEOLAR_L3)}):")
for l in PULMONARY_ALVEOLAR_L3: print(f"  {l}")

print(f"\nL2 category summary:")
print(annot_df[annot_df['action'] != 'DROP']['L2'].value_counts().to_string())


Seed set to 42



DROP clusters (4):
  Endothelia_vascular_Cap_a_c0
  Endothelia_vascular_venous_systemic_c3
  Endothelia_vascular_venous_systemic_c5
  Fibro_myofibroblast_c0

REASSIGN clusters (3):
  Endothelia_vascular_venous_systemic_c4 -> L2='Fibroblast', L3='Fibro_stress_activated'
  Fibro_adventitial_c2 -> L2='Fibroblast', L3='Fibro_alveolar'
  Muscle_perivascular_immune_recruiting_c1 -> L2='Pericyte', L3='Muscle_pericyte_systemic'

Pulmonary/Alveolar L3 labels for tissue-aware scANVI (5):
  Endothelia_vascular_arterial_pulmonary
  Endothelia_vascular_venous_pulmonary
  Fibro_alveolar
  Muscle_pericyte_pulmonary
  Muscle_smooth_pulmonary

L2 category summary:
L2
Endothelial      20
Fibroblast       11
Pericyte          7
Smooth_Muscle     5
Schwann           1


## Step 1: LOAD DATA

In [2]:
# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================
print("\n" + "="*70 + "\nSTEP 1: LOAD DATA\n" + "="*70)
adata = sc.read_h5ad(INPUT_H5AD)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"Obs keys: {list(adata.obs.columns)}")

for col in [SRC_L2_COL, BATCH_KEY]:
    assert col in adata.obs.columns, f"[ERROR] Missing: {col}"
assert SRC_L3_COL in adata.obs.columns, f"[ERROR] Missing configured L3 column: {SRC_L3_COL}"

# Auto-detect best column matching ANNOTATION_TABLE cluster IDs
def _annotation_match_count(values):
    return pd.Index(values).isin(annot_df.index).sum()

l3_str   = adata.obs[SRC_L3_COL].astype(str)
l3_uniq  = l3_str.unique()
n_match  = _annotation_match_count(l3_uniq)
best_col = SRC_L3_COL

candidate_cols = []
if 'subcluster_id' in adata.obs.columns:
    candidate_cols.append('subcluster_id')
candidate_cols.extend([
    c for c in adata.obs.columns
    if c not in (SRC_L3_COL, SRC_L2_COL) and ('l3' in c.lower() or 'subcluster' in c.lower())
])

for c in dict.fromkeys(candidate_cols):
    vals = adata.obs[c].astype(str).unique()
    m = _annotation_match_count(vals)
    if m > n_match:
        best_col = c; l3_uniq = vals; n_match = m

# Fallback: reconstruct fine cluster IDs from L2 prefix + subcluster_id suffix
if n_match == 0 and SRC_L2_COL in adata.obs.columns and 'subcluster_id' in adata.obs.columns:
    reconstructed_col = '__reconstructed_cluster_id'
    reconstructed = (
        adata.obs[SRC_L2_COL].astype(str)
        + '_c'
        + adata.obs['subcluster_id'].astype(str)
    )
    adata.obs[reconstructed_col] = pd.Categorical(reconstructed)
    vals = adata.obs[reconstructed_col].astype(str).unique()
    m = _annotation_match_count(vals)
    if m > n_match:
        print(f"[INFO] Reconstructed cluster IDs from '{SRC_L2_COL}' + 'subcluster_id' "
              f"-> '{reconstructed_col}' (coverage {m}/{len(vals)})")
        best_col = reconstructed_col; l3_uniq = vals; n_match = m

if best_col != SRC_L3_COL and n_match > 0:
    print(f"[INFO] Auto-switch source L3 column: '{SRC_L3_COL}' -> '{best_col}' "
          f"(coverage {n_match}/{len(l3_uniq)})")
    SRC_L3_COL = best_col

if n_match == 0:
    raise ValueError(
        f"[ERROR] No values in '{SRC_L3_COL}' match ANNOTATION_TABLE. "
        "Check cluster ID format or update SRC_L3_COL."
    )

TISSUE_KEY_AVAILABLE = TISSUE_KEY in adata.obs.columns
if not TISSUE_KEY_AVAILABLE:
    similar = [c for c in adata.obs.columns if any(k in c.lower() for k in ('tissue','organ','site'))]
    print(f"[WARN] TISSUE_KEY='{TISSUE_KEY}' not found. Candidates: {similar}")
    print(f"       Tissue-aware Unknown assignment will be SKIPPED.")
else:
    print(f"\nTissue distribution:\n{adata.obs[TISSUE_KEY].value_counts().to_string()}")

not_in_table = [x for x in l3_uniq if x not in annot_df.index]
print(f"\nAnnotation table coverage: {len(l3_uniq)-len(not_in_table)}/{len(l3_uniq)}")
if not_in_table:
    print(f"  [WARN] Unmatched (first 20): {not_in_table[:20]}")



STEP 1: LOAD DATA


Loaded: 60,828 cells x 36,789 genes
Obs keys: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'plateID', 'status', 'donorID', 'cDate', 'age', 'sex', 'cellType', 'percent.mt', 'percent.ribo', 'tissue', 'tissue_sampling_method', 'dataset', 'sample', 'percent.rb', 'decontX_contamination', 'decontX_clusters', 'nCount_decontXcounts', 'nFeature_decontXcounts', 'donor_id', 'Group', 'Ethnicity_inferred', 'Smoker', 'COVID_status', 'First_symptoms_collection_interval', 'Kit_version', 'batch', 'log1p_n_genes', 'percent_total_sarscov2', 'n_counts_sarscov2', 'scrublet_score', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'Source', 'Location', 'CellType', 'BroadCellType', 'organism_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'ag

## Step 2: ASSIGN L1 / L2 / L3 FROM ANNOTATION TABLE

In [3]:
# ============================================================================
# STEP 2: ASSIGN L1 / L2 / L3 FROM ANNOTATION TABLE
# ============================================================================
# L1 = 'Stromal' for all non-contaminant cells (assigned after DROP removal).
# L2 = broad class from table (Endothelial / Fibroblast / Pericyte / Smooth_Muscle / Schwann).
# L3 = cluster prefix without _c{N}; REASSIGN clusters get overridden L3 in table.
# Action column is stored as 'annot_action' for traceability.
# ============================================================================
print("\n" + "="*70 + "\nSTEP 2: ASSIGN L1/L2/L3 FROM ANNOTATION TABLE\n" + "="*70)

l3_str = adata.obs[SRC_L3_COL].astype(str)

l2_mapped     = l3_str.map(annot_df['L2'])
l3_mapped     = l3_str.map(annot_df['L3'])
action_mapped = l3_str.map(annot_df['action'])

n_mapped = l2_mapped.notna().sum()
print(f"Annotation-mapped cells: {n_mapped:,}/{adata.n_obs:,}")
if n_mapped == 0:
    raise ValueError(
        f"[ERROR] Annotation mapping produced 0 matched cells from '{SRC_L3_COL}'. "
        "Stopping here to avoid silent all-fallback downstream labels."
    )

# L1 = 'Stromal'; contaminants get 'Contaminants' (removed in Step 3)
adata.obs[OUT_L1]         = np.where(l2_mapped.fillna('Unknown') == 'Contaminants',
                                      'Contaminants', 'Stromal')
adata.obs[OUT_L2]         = l2_mapped.fillna('Unresolved')
adata.obs[OUT_L3]         = l3_mapped.fillna('Unresolved')
adata.obs['annot_action'] = action_mapped.fillna('REVIEW')

print("\nAction distribution:")
print(adata.obs['annot_action'].value_counts().to_string())
print("\nL2 distribution (including contaminants):")
print(adata.obs[OUT_L2].value_counts().to_string())
print("\nL3 distribution (top 30):")
print(adata.obs[OUT_L3].value_counts().head(30).to_string())

if n_mapped < adata.n_obs:
    unmatched = pd.Index(l3_str[l2_mapped.isna()].unique()).tolist()
    print(f"\n[WARN] Unmatched {SRC_L3_COL} values (first 20): {unmatched[:20]}")



STEP 2: ASSIGN L1/L2/L3 FROM ANNOTATION TABLE
Annotation-mapped cells: 60,828/60,828

Action distribution:
annot_action
KEEP        32717
REVIEW      18352
REASSIGN     5625
DROP         4134

L2 distribution (including contaminants):
cell_type_L2
Endothelial      34364
Fibroblast       17851
Contaminants      4134
Smooth_Muscle     2372
Pericyte          1967
Schwann            140

L3 distribution (top 30):
cell_type_L3
Endothelia_vascular_venous_systemic       17319
Fibro_adventitial                          7665
Endothelia_Lymphatic                       6279
Endothelia_vascular_Cap_g                  6067
Fibro_alveolar                             4794
Contaminants                               4134
Fibro_peribronchial                        3500
Endothelia_vascular_venous_pulmonary       1775
Fibro_stress_activated                     1677
Endothelia_vascular_arterial_pulmonary     1662
Muscle_smooth_pulmonary                    1427
Muscle_pericyte_pulmonary                  11

## Step 3: REMOVE CONTAMINATION CLUSTERS

In [4]:
# ============================================================================
# STEP 3: REMOVE CONTAMINATION CLUSTERS
# ============================================================================
# L2/L3 assignment (including REASSIGN overrides) was done in Step 2.
# DROP clusters are identified by action='DROP'; no manual reassignment needed.
# ============================================================================
print("\n" + "="*70 + "\nSTEP 3: REMOVE CONTAMINATION CLUSTERS\n" + "="*70)
n_before  = adata.n_obs
drop_mask = adata.obs[SRC_L3_COL].astype(str).isin(DROP_CLUSTERS)
for c in DROP_CLUSTERS:
    n = (adata.obs[SRC_L3_COL].astype(str) == c).sum()
    l3_label = annot_df.loc[c, 'L3'] if c in annot_df.index else 'N/A'
    print(f"  {c}: {n:,}  [L3={l3_label!r}]")
adata = adata[~drop_mask].copy()
print(f"\n{n_before:,} -> {adata.n_obs:,} cells  (removed {n_before - adata.n_obs:,})")

# Verify reassigned clusters are present and correctly labeled
print("\nREASSIGN verification:")
for c in REASSIGN_CLUSTERS:
    mask = adata.obs[SRC_L3_COL].astype(str) == c
    if mask.sum() == 0:
        print(f"  [WARN] {c}: 0 cells found (was it in DROP list?)")
    else:
        sample_l2 = adata.obs.loc[mask, OUT_L2].iloc[0]
        sample_l3 = adata.obs.loc[mask, OUT_L3].iloc[0]
        print(f"  {c}: {mask.sum():,} cells -> L2={sample_l2!r}, L3={sample_l3!r}")

gc.collect()



STEP 3: REMOVE CONTAMINATION CLUSTERS
  Endothelia_vascular_Cap_a_c0: 265  [L3='Contaminants']
  Endothelia_vascular_venous_systemic_c3: 2,764  [L3='Contaminants']
  Endothelia_vascular_venous_systemic_c5: 705  [L3='Contaminants']
  Fibro_myofibroblast_c0: 400  [L3='Contaminants']

60,828 -> 56,694 cells  (removed 4,134)

REASSIGN verification:
  Endothelia_vascular_venous_systemic_c4: 1,677 cells -> L2='Fibroblast', L3='Fibro_stress_activated'
  Fibro_adventitial_c2: 3,558 cells -> L2='Fibroblast', L3='Fibro_alveolar'
  Muscle_perivascular_immune_recruiting_c1: 390 cells -> L2='Pericyte', L3='Muscle_pericyte_systemic'


6362

## Step 4: VERIFY DATA LAYERS

In [5]:
# ============================================================================
# STEP 4: VERIFY DATA LAYERS
# ============================================================================
print("\n" + "="*70 + "\nSTEP 4: VERIFY DATA LAYERS\n" + "="*70)
if 'counts' not in adata.layers:
    if adata.raw is not None:
        print("[WARN] 'counts' layer missing -- inferring from .raw.X")
        missing = adata.var_names.difference(adata.raw.var_names)
        if len(missing) > 0:
            raise ValueError(f".raw is missing {len(missing)} current genes; cannot recover counts safely.")
        adata.layers['counts'] = sparse.csr_matrix(
            adata.raw[:, adata.var_names].X
        ).astype(np.float32)
    else:
        raise ValueError("No 'counts' layer and no .raw -- cannot continue.")
if 'log1p' not in adata.layers:
    adata.layers['log1p'] = adata.X.copy()
adata.X = adata.layers['log1p']
for lk in ['counts', 'log1p']:
    if lk in adata.layers and not sparse.issparse(adata.layers[lk]):
        adata.layers[lk] = sparse.csr_matrix(adata.layers[lk])
if not sparse.issparse(adata.X):
    adata.X = sparse.csr_matrix(adata.X)
print(f"[OK] shape={adata.shape}, counts={adata.layers['counts'].data.nbytes/1e9:.2f} GB")



STEP 4: VERIFY DATA LAYERS
[OK] shape=(56694, 36789), counts=0.98 GB


## Step 5: FILTER SMALL BATCHES

In [6]:
# ============================================================================
# STEP 5: FILTER SMALL BATCHES
# ============================================================================
print("\n" + "="*70 + "\nSTEP 5: FILTER SMALL BATCHES\n" + "="*70)
bc = adata.obs[BATCH_KEY].value_counts()
small = bc[bc < MIN_CELLS_BATCH].index.tolist()
if small:
    n_pre = adata.n_obs
    adata = adata[~adata.obs[BATCH_KEY].isin(small)].copy()
    print(f"Removed {len(small)} small batches: {n_pre:,} -> {adata.n_obs:,}")
else:
    print(f"[OK] All {adata.obs[BATCH_KEY].nunique()} batches pass minimum size")
gc.collect()



STEP 5: FILTER SMALL BATCHES
Removed 3 small batches: 56,694 -> 56,689


6265

## Step 6: HVG SELECTION + FULL-GENE .raw (shared memory, zero extra cost)

In [7]:
# ============================================================================
# STEP 6: HVG SELECTION + PRESERVE .raw (shared memory, zero extra cost)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 6: HVG SELECTION + PRESERVE .raw\n" + "="*70)
print(f"Selecting {N_HVG} HVGs from {adata.n_vars:,} genes...")
try:
    sc.pp.highly_variable_genes(adata, layer='counts', n_top_genes=N_HVG,
                                 batch_key=BATCH_KEY, flavor='seurat_v3', subset=False)
    hvg_method = "batch-aware seurat_v3"
except Exception as e:
    print(f"  [WARN] {e}")
    try:
        sc.pp.highly_variable_genes(adata, layer='counts', n_top_genes=N_HVG,
                                     flavor='seurat_v3', subset=False)
        hvg_method = "seurat_v3 (no batch)"
    except Exception as e2:
        print(f"  [WARN] {e2}")
        sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, subset=False)
        hvg_method = "default"
print(f"HVG method: {hvg_method} | HVGs: {adata.var['highly_variable'].sum():,}")

# CRITICAL: preserve full genes to .raw BEFORE subsetting (shared memory, no copy)
print(f"Saving {adata.n_vars:,}-gene data to .raw (shared memory)...")
adata.raw = sc.AnnData(
    X   = adata.layers['counts'],   # shared memory -- no .copy()
    obs = adata.obs.copy(),
    var = adata.var.copy()
)
adata = adata[:, adata.var['highly_variable']].copy()

# v1.1 FIX [4]: recover counts from .raw (full-gene counts), not .X (log1p)
if 'counts' not in adata.layers:
    print("[WARN] counts layer missing after HVG subset -- recovering from .raw")
    adata.layers['counts'] = sparse.csr_matrix(
        adata.raw[:, adata.var_names].X
    ).astype(np.float32)

print(f"HVG subset: {adata.shape} | .raw: {adata.raw.n_vars:,} genes (full)")
gc.collect()



STEP 6: HVG SELECTION + PRESERVE .raw
Selecting 4000 HVGs from 36,789 genes...
extracting highly variable genes
  [WARN] b'There are other near singularities as well. 0.090619\n'
extracting highly variable genes
HVG method: seurat_v3 (no batch) | HVGs: 4,000
Saving 36,789-gene data to .raw (shared memory)...
HVG subset: (56689, 4000) | .raw: 36,789 genes (full)


6392

## Step 7: scVI TRAINING (re-integration on cleaned data)

In [8]:
# ============================================================================
# STEP 7: scVI TRAINING (re-integration on cleaned data)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 7: scVI TRAINING\n" + "="*70)
SCVI_MODEL_PATH = MODEL_DIR / "scvi_stromal_v1_3"
scvi.model.SCVI.setup_anndata(adata, layer='counts', batch_key=BATCH_KEY)
model_scvi = scvi.model.SCVI(
    adata,
    n_latent=N_LATENT, n_hidden=N_HIDDEN,
    n_layers=N_LAYERS, dropout_rate=DROPOUT,
    gene_likelihood='nb', dispersion='gene-batch'
)
print(f"n_latent={N_LATENT}, cells={adata.n_obs:,}, HVGs={adata.n_vars:,}")

train_accelerator = 'gpu' if USE_GPU else 'cpu'
train_devices = 1
print(f"Training backend: accelerator={train_accelerator}, devices={train_devices}")

t0 = time.time()
model_scvi.train(
    max_epochs=MAX_EPOCHS_SCVI,
    batch_size=BATCH_SIZE,
    early_stopping=True,
    early_stopping_patience=20,
    train_size=0.9,
    accelerator=train_accelerator,
    devices=train_devices,
    plan_kwargs={'lr': 1e-3}
)
print(f"[OK] scVI done in {(time.time()-t0)/60:.1f} min")
model_scvi.save(str(SCVI_MODEL_PATH), overwrite=True)
pd.Series(adata.var_names.tolist()).to_csv(
    SCVI_MODEL_PATH / "hvg_genes.csv", index=False, header=False
)
adata.obsm['X_scvi'] = model_scvi.get_latent_representation()
print(f"[OK] X_scvi: {adata.obsm['X_scvi'].shape}")


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



STEP 7: scVI TRAINING
n_latent=75, cells=56,689, HVGs=4,000
Training backend: accelerator=gpu, devices=1


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 400/400: 100%|██████████| 400/400 [40:42<00:00,  6.36s/it, v_num=1, train_loss_step=852, train_loss_epoch=748]    

`Trainer.fit` stopped: `max_epochs=400` reached.


Epoch 400/400: 100%|██████████| 400/400 [40:42<00:00,  6.11s/it, v_num=1, train_loss_step=852, train_loss_epoch=748]
[OK] scVI done in 40.7 min
[OK] X_scvi: (56689, 75)


## Step 8: NEIGHBORS + UMAP (scVI latent)

In [9]:
# ============================================================================
# STEP 8: NEIGHBORS + UMAP (scVI latent)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 8: NEIGHBORS + UMAP (scVI)\n" + "="*70)
sc.pp.neighbors(adata, use_rep='X_scvi', n_neighbors=30,
                random_state=RANDOM_SEED, key_added='neighbors_scvi')
sc.tl.umap(adata, neighbors_key='neighbors_scvi', random_state=RANDOM_SEED)
adata.obsm['X_umap_scvi'] = adata.obsm['X_umap'].copy()

sc.settings.vector_friendly = True   # rasterizes scatter points in all subsequent sc.pl calls

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sc.pl.embedding(adata, basis='umap', color=OUT_L2, ax=axes[0],
                show=False, frameon=False, size=2,
                legend_loc='right margin', legend_fontsize=8,
                title='Post-scVI UMAP: L2 (broad class)')
sc.pl.embedding(adata, basis='umap', color=OUT_L3, ax=axes[1],
                show=False, frameon=False, size=2,
                legend_loc='right margin', legend_fontsize=6,
                title='Post-scVI UMAP: L3 (cluster prefix)')
plt.tight_layout()
fig.savefig(FIG_DIR / 'scvi_umap_L2_L3.pdf', dpi=300, bbox_inches='tight')
plt.close('all'); gc.collect()
print("[OK] Saved: scvi_umap_L2_L3.pdf")


STEP 8: NEIGHBORS + UMAP (scVI)
computing neighbors
    finished (0:01:16)
computing UMAP
    finished (0:01:56)
[OK] Saved: scvi_umap_L2_L3.pdf


## Step 9: BUILD scANVI LABELS (tissue-aware, L3-based)

In [10]:
# ============================================================================
# STEP 9: BUILD scANVI LABELS (tissue-aware, L3-based)
# ============================================================================
# scANVI training label = L3 (cluster prefix without _c{N}).
# This merges all subclusters within the same anatomical type into one label,
# giving scANVI more cells per class and more stable semi-supervised gradients.
#
# Tissue-aware Unknown assignment:
#   Cells with L3 in PULMONARY_ALVEOLAR_L3 from non-lung/trachea tissue -> Unknown.
#   Rationale: alveolar fibroblasts, pulmonary pericytes, aCap endothelia etc. are
#   lung-specific. Labeling nasal/sinus cells with these L3 names would cause scANVI
#   to learn spurious cross-tissue associations.
#
# kNN purity filter (k=30, threshold=0.5):
#   Low-purity labeled cells (neighbors majority disagree on label) -> Unknown.
#   Applied in scVI latent space; vectorized for efficiency.
# ============================================================================
print("\n" + "="*70 + "\nSTEP 9: BUILD scANVI TRAINING LABELS (L3-based)\n" + "="*70)

# Base: L3 label for all retained cells
scanvi_labels = adata.obs[OUT_L3].astype(str).copy()

# Tissue-aware Unknown assignment
if TISSUE_KEY_AVAILABLE:
    tissue_str  = adata.obs[TISSUE_KEY].astype(str)
    tissue_norm = (
        tissue_str.str.strip().str.lower()
        .str.replace(r'[_-]+', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
    )
    lung_trachea_norm = {
        x.strip().lower().replace('_', ' ').replace('-', ' ')
        for x in LUNG_TRACHEA_VALUES
    }
    exact_tissue_match   = tissue_norm.isin(lung_trachea_norm)
    keyword_tissue_match = tissue_norm.str.contains(
        r'\blung\b|\btrachea\b|\bairway\b|\bbronch|\bparenchyma\b|\bpulmon',
        regex=True, na=False
    )
    is_lung_or_trachea = exact_tissue_match | keyword_tissue_match

    auto_extra = sorted(tissue_str[keyword_tissue_match & ~exact_tissue_match].unique().tolist())
    if auto_extra:
        print(f"[INFO] Additional tissue values auto-matched as lung/trachea: {auto_extra[:20]}")

    # Check L3 label (not original fine cluster) for pulmonary/alveolar membership
    is_pulm_alv     = scanvi_labels.isin(PULMONARY_ALVEOLAR_L3)
    tissue_mismatch = is_pulm_alv & ~is_lung_or_trachea
    n_mismatch      = tissue_mismatch.sum()
    print(f"Lung/trachea cells:              {is_lung_or_trachea.sum():,}")
    print(f"Pulmonary/alveolar L3 cells:     {is_pulm_alv.sum():,}")
    print(f"Tissue mismatch -> Unknown:      {n_mismatch:,}")
    if n_mismatch > 0:
        detail = (
            adata.obs.loc[tissue_mismatch, [OUT_L3, TISSUE_KEY]]
            .value_counts().reset_index(name='n_cells')
        )
        print(f"\nMismatch detail:\n{detail.head(20).to_string(index=False)}")
    scanvi_labels[tissue_mismatch] = UNLABELED
else:
    print("[WARN] Tissue key unavailable -- skipping tissue-aware filtering.")

# Vectorized kNN purity filter in scVI latent space
print(f"\nComputing kNN purity (k={PURITY_K}) in scVI latent space...")
knn = NearestNeighbors(n_neighbors=PURITY_K, algorithm='auto', n_jobs=16)
knn.fit(adata.obsm['X_scvi'])
_, knn_idx = knn.kneighbors(adata.obsm['X_scvi'])
labels_arr     = scanvi_labels.values                                   # (n_obs,)
neighbor_labels = labels_arr[knn_idx]                                   # (n_obs, k)
purity = np.mean(neighbor_labels == labels_arr[:, None], axis=1).astype(np.float32)
is_unknown_mask = labels_arr == UNLABELED
purity[is_unknown_mask] = 1.0                                           # Unknown cells skip purity eval
adata.obs['label_purity_scanvi'] = purity

low_purity = (purity < PURITY_THRESHOLD) & ~is_unknown_mask
scanvi_labels[low_purity] = UNLABELED
print(f"  Low purity -> Unknown: {low_purity.sum():,} ({low_purity.sum()/adata.n_obs*100:.1f}%)")
del knn, knn_idx, neighbor_labels; gc.collect()

adata.obs['scanvi_label'] = scanvi_labels.values
n_labeled = (scanvi_labels != UNLABELED).sum()
n_unknown = (scanvi_labels == UNLABELED).sum()
print(f"\nLabel summary: {n_labeled:,} labeled | {n_unknown:,} Unknown")
print("Label distribution (top 30):")
for lbl, n in scanvi_labels.value_counts().head(30).items():
    mark = '  <-- UNLABELED' if lbl == UNLABELED else ''
    print(f"  {lbl}: {n:,}{mark}")



STEP 9: BUILD scANVI TRAINING LABELS (L3-based)
[INFO] Additional tissue values auto-matched as lung/trachea: ['lung parenchyma', 'respiratory airway']
Lung/trachea cells:              18,705
Pulmonary/alveolar L3 cells:     10,827
Tissue mismatch -> Unknown:      3,558

Mismatch detail:
           cell_type_L3 tissue  n_cells
         Fibro_alveolar   nose     2231
         Fibro_alveolar  sinus      687
Muscle_smooth_pulmonary   nose      629
Muscle_smooth_pulmonary  sinus       11

Computing kNN purity (k=30) in scVI latent space...
  Low purity -> Unknown: 13,124 (23.2%)

Label summary: 40,007 labeled | 16,682 Unknown
Label distribution (top 30):
  Unknown: 16,682  <-- UNLABELED
  Endothelia_vascular_venous_systemic: 15,789
  Endothelia_Lymphatic: 5,577
  Fibro_adventitial: 4,780
  Endothelia_vascular_Cap_g: 3,948
  Endothelia_vascular_arterial_pulmonary: 1,489
  Endothelia_vascular_venous_pulmonary: 1,444
  Fibro_peribronchial: 1,263
  Fibro_alveolar: 1,229
  Muscle_pericyte_pulm

## Step 10: scANVI TRAINING

In [11]:
# ============================================================================
# STEP 10: scANVI TRAINING
# ============================================================================
print("\n" + "="*70 + "\nSTEP 10: scANVI TRAINING\n" + "="*70)
SCANVI_MODEL_PATH = MODEL_DIR / "scanvi_stromal_v1_3"

_labels = adata.obs['scanvi_label'].astype(str)
_labeled_mask = _labels != UNLABELED
_n_labeled_classes = _labels[_labeled_mask].nunique()
print(f"Labeled classes (excluding '{UNLABELED}'): {_n_labeled_classes}")

if _n_labeled_classes < 2:
    print("[WARN] <2 labeled classes; skipping scANVI and using scVI fallback outputs.")
    adata.obsm['X_scanvi'] = adata.obsm['X_scvi'].copy()
    adata.obs['cell_type_scanvi_pred'] = _labels.copy()
    adata.obs['scanvi_uncertainty'] = np.where(
        _labels == UNLABELED, 1.0, 0.0
    ).astype(np.float32)
    SCANVI_TRAINED = False
    print(f"[OK] Fallback X_scanvi: {adata.obsm['X_scanvi'].shape}")
else:
    # v1.1 FIX [1]: from_scvi_model inherits data manager; no separate setup_anndata needed
    model_scanvi = scvi.model.SCANVI.from_scvi_model(
        model_scvi,
        unlabeled_category=UNLABELED,
        labels_key='scanvi_label'
    )

    # v1.1 FIX [3]: release scVI model before scANVI training to free GPU memory
    del model_scvi
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"[OK] GPU cache cleared before scANVI training")

    train_accelerator = 'gpu' if USE_GPU else 'cpu'
    train_devices = 1
    import inspect
    scanvi_train_kwargs = dict(
        max_epochs=MAX_EPOCHS_SCANVI,
        batch_size=BATCH_SIZE,
        early_stopping=True,
        early_stopping_patience=20,
        train_size=0.9,
        accelerator=train_accelerator,
        devices=train_devices,
    )
    if 'n_samples_per_label' in inspect.signature(model_scanvi.train).parameters:
        scanvi_train_kwargs['n_samples_per_label'] = 100

    t0 = time.time()
    model_scanvi.train(**scanvi_train_kwargs)
    print(f"[OK] scANVI done in {(time.time()-t0)/60:.1f} min")
    model_scanvi.save(str(SCANVI_MODEL_PATH), overwrite=True)

    # Save var_names for scArches compatibility
    pd.Series(adata.var_names.tolist()).to_csv(
        SCANVI_MODEL_PATH / "SCANVI_var_names.csv", index=False, header=False
    )

    adata.obsm['X_scanvi']             = model_scanvi.get_latent_representation()
    adata.obs['cell_type_scanvi_pred'] = model_scanvi.predict()
    soft_pred     = model_scanvi.predict(soft=True)
    soft_pred_arr = soft_pred.to_numpy() if hasattr(soft_pred, 'to_numpy') else np.asarray(soft_pred)
    adata.obs['scanvi_uncertainty'] = (1 - soft_pred_arr.max(axis=1)).astype(np.float32)
    SCANVI_TRAINED = True
    print(f"[OK] X_scanvi: {adata.obsm['X_scanvi'].shape}")
    print("scANVI prediction distribution (L3-level):")
    print(adata.obs['cell_type_scanvi_pred'].value_counts().head(30).to_string())



STEP 10: scANVI TRAINING
Labeled classes (excluding 'Unknown'): 18
[OK] GPU cache cleared before scANVI training
INFO     Training for 200 epochs.                                                                                  


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 200/200: 100%|██████████| 200/200 [43:15<00:00, 13.99s/it, v_num=1, train_loss_step=854, train_loss_epoch=738]

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 200/200: 100%|██████████| 200/200 [43:15<00:00, 12.98s/it, v_num=1, train_loss_step=854, train_loss_epoch=738]
[OK] scANVI done in 43.3 min
[OK] X_scanvi: (56689, 75)
scANVI prediction distribution (L3-level):
cell_type_scanvi_pred
Endothelia_vascular_venous_systemic       8027
Endothelia_vascular_Cap_g                 6794
Endothelia_Lymphatic                      6617
Fibro_adventitial                         5972
Endothelia_vascular_Cap_a                 4385
Fibro_peribronchial                       4073
Endothelia_vascular_venous_pulmonary      2891
Fibro_stress_activated                    2517
Fibro_alveolar                            2265
Schwann_nonmyelinating                    2005
Muscle_perivascular_immune_recruiting     1961
Muscle_smooth_pulmonary                   1860
Endothelia_vascular_arterial_pulmonary    1735
Endothelia_vascular_arterial_systemic     1508
Muscle_pericyte_pulmonary                 1387
Fibro_myofibroblast                       1300
Muscle_per

## Step 11: UMAP + FIGURES (scANVI latent)

In [12]:
# ============================================================================
# STEP 11: UMAP + FIGURES (scANVI latent)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 11: UMAP (scANVI) + FIGURES\n" + "="*70)
sc.pp.neighbors(adata, use_rep='X_scanvi', n_neighbors=30,
                random_state=RANDOM_SEED, key_added='neighbors_scanvi')
sc.tl.umap(adata, neighbors_key='neighbors_scanvi', random_state=RANDOM_SEED)
adata.obsm['X_umap_scanvi'] = adata.obsm['X_umap'].copy()

sc.settings.vector_friendly = True   # rasterizes scatter points in all subsequent sc.pl calls

# Main overview: L2 | L3 | scANVI prediction
fig, axes = plt.subplots(1, 3, figsize=(27, 8))
sc.pl.embedding(adata, basis='umap', color=OUT_L2, ax=axes[0],
                show=False, frameon=False, size=2,
                legend_loc='right margin', legend_fontsize=8,
                title='L2 Broad Class')
sc.pl.embedding(adata, basis='umap', color='scanvi_label', ax=axes[1],
                show=False, frameon=False, size=2,
                legend_loc='right margin', legend_fontsize=6,
                title='scANVI Training Label (L3)')
sc.pl.embedding(adata, basis='umap', color='cell_type_scanvi_pred', ax=axes[2],
                show=False, frameon=False, size=2,
                legend_loc='right margin', legend_fontsize=6,
                title='scANVI Prediction (L3)')
plt.suptitle('Stromal/Vascular Post-scANVI (v1.3)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_DIR / 'scanvi_umap_overview.pdf', dpi=300, bbox_inches='tight')
plt.close('all'); print("[OK] Saved: scanvi_umap_overview.pdf")

# Uncertainty map
fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.embedding(adata, basis='umap', color='scanvi_uncertainty',
                ax=ax, show=False, frameon=False, size=2,
                cmap='RdYlBu_r', vmin=0, vmax=0.5,
                title='scANVI Prediction Uncertainty')
fig.savefig(FIG_DIR / 'scanvi_uncertainty_umap.pdf', dpi=300, bbox_inches='tight')
plt.close('all'); print("[OK] Saved: scanvi_uncertainty_umap.pdf")

if TISSUE_KEY_AVAILABLE:
    fig, ax = plt.subplots(figsize=(9, 7))
    sc.pl.embedding(adata, basis='umap', color=TISSUE_KEY, ax=ax,
                    show=False, frameon=False, size=2, title='Tissue Source')
    fig.savefig(FIG_DIR / 'scanvi_umap_tissue.pdf', dpi=300, bbox_inches='tight')
    plt.close('all'); print("[OK] Saved: scanvi_umap_tissue.pdf")
gc.collect()


STEP 11: UMAP (scANVI) + FIGURES
computing neighbors
    finished (0:00:15)
computing UMAP
    finished (0:01:52)
[OK] Saved: scanvi_umap_overview.pdf
[OK] Saved: scanvi_uncertainty_umap.pdf
[OK] Saved: scanvi_umap_tissue.pdf


15871

## Step 12: LEIDEN CLUSTERING (scANVI latent)

In [13]:
# ============================================================================
# STEP 12: LEIDEN CLUSTERING (scANVI latent)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 12: LEIDEN CLUSTERING (scANVI latent)\n" + "="*70)
for res in [0.4, 0.6, 0.8, 1.0]:
    key = f'leiden_scanvi_r{res:.1f}'
    sc.tl.leiden(adata, resolution=res, neighbors_key='neighbors_scanvi',
                 key_added=key, random_state=RANDOM_SEED)
    print(f"  Resolution {res}: {adata.obs[key].nunique()} clusters")



STEP 12: LEIDEN CLUSTERING (scANVI latent)
running Leiden clustering
    finished (0:01:20)
  Resolution 0.4: 14 clusters
running Leiden clustering
    finished (0:01:06)
  Resolution 0.6: 16 clusters
running Leiden clustering
    finished (0:01:29)
  Resolution 0.8: 19 clusters
running Leiden clustering
    finished (0:01:30)
  Resolution 1.0: 20 clusters


## Step 13: SAVE FINAL OUTPUT

In [14]:
# ============================================================================
# STEP 13: SAVE FINAL OUTPUT
# ============================================================================
print("\n" + "="*70 + "\nSTEP 13: SAVE OUTPUT\n" + "="*70)
adata.uns['stromal_reintegration_params'] = {
    'version':           '1.3',
    'date':              '2026-03-08',
    'hierarchy':         {'L1': 'Stromal', 'L2': 'broad_class', 'L3': 'cluster_prefix_no_cN'},
    'n_cells_input':     int(n_before),
    'n_cells_output':    int(adata.n_obs),
    'drop_clusters':     DROP_CLUSTERS,
    'reassign_clusters': REASSIGN_CLUSTERS,
    'n_hvg':             int(N_HVG),
    'hvg_method':        hvg_method,
    'n_latent':          N_LATENT,
    'batch_key':         BATCH_KEY,
    'tissue_key':        TISSUE_KEY,
    'tissue_key_available': TISSUE_KEY_AVAILABLE,
    'lung_trachea_values': sorted(LUNG_TRACHEA_VALUES),
    'pulmonary_alveolar_l3': PULMONARY_ALVEOLAR_L3,
    'purity_k':          PURITY_K,
    'purity_threshold':  PURITY_THRESHOLD,
    'scanvi_trained':    SCANVI_TRAINED,
    'random_seed':       RANDOM_SEED,
    'src_l3_column_used': SRC_L3_COL,
    'fixes': [
        'v1.1_scanvi_setup_anndata_removed',
        'v1.1_knn_purity_vectorized',
        'v1.1_scvi_model_released_before_scanvi',
        'v1.1_counts_fallback_from_raw',
        'v1.3_hierarchy_redesign_L1_L2_L3',
        'v1.3_scanvi_label_uses_L3_not_coarse_L3',
    ],
}
print(f"Cells: {adata.n_obs:,} | HVGs: {adata.n_vars:,} | .raw: {adata.raw.n_vars:,}")
adata.write_h5ad(OUTPUT_H5AD, compression='gzip', compression_opts=9)

pd.Series(adata.var_names.tolist()).to_csv(
    OUTPUT_DIR / 'hvg_genes_final.csv', index=False, header=False
)
pd.Series(adata.raw.var_names.tolist()).to_csv(
    OUTPUT_DIR / 'all_genes_raw.csv', index=False, header=False
)
adata.obs[
    [OUT_L1, OUT_L2, OUT_L3, SRC_L3_COL, 'annot_action',
     'scanvi_label', 'label_purity_scanvi',
     'cell_type_scanvi_pred', 'scanvi_uncertainty']
].to_csv(OUTPUT_DIR / 'scanvi_label_summary.csv')
print(f"[OK] {OUTPUT_H5AD}")

# ---- Pipeline run log ----
elapsed = time.time() - PIPELINE_START
log_lines = [
    f"Pipeline:       stromal_reintegration v1.3",
    f"Date:           2026-03-08",
    f"Runtime:        {elapsed/60:.1f} min",
    f"Cells in:       {n_before:,}",
    f"Cells out:      {adata.n_obs:,}",
    f"HVGs:           {adata.n_vars:,}",
    f"HVG method:     {hvg_method}",
    f"scANVI trained: {SCANVI_TRAINED}",
    f"Labeled:        {n_labeled:,}",
    f"Unknown:        {n_unknown:,}",
    f"Output h5ad:    {OUTPUT_H5AD}",
]
with open(OUTPUT_DIR / 'pipeline_run_log.txt', 'w') as f:
    f.write('\n'.join(log_lines) + '\n')

# ---- AnnData structure summary ----
struct_lines = [
    f"AnnData structure: {adata.shape}",
    f"  .X             : log1p normalized (sparse)",
    f"  .layers['counts']: raw counts (sparse)",
    f"  .layers['log1p']: log1p (sparse)",
    f"  .raw.X         : full-gene counts ({adata.raw.n_vars:,} genes)",
    f"  .obs columns   : {list(adata.obs.columns)}",
    f"  .obsm keys     : {list(adata.obsm.keys())}",
    f"  cell_type_L1   : {adata.obs[OUT_L1].value_counts().to_dict()}",
    f"  cell_type_L2   : {adata.obs[OUT_L2].value_counts().to_dict()}",
    f"  cell_type_L3   : {adata.obs[OUT_L3].nunique()} unique values",
]
with open(OUTPUT_DIR / 'anndata_structure.txt', 'w') as f:
    f.write('\n'.join(struct_lines) + '\n')

print(f"\n{'='*70}\nPIPELINE COMPLETE  ({elapsed/60:.1f} min)\n{'='*70}")
print(f"Cells: {n_before:,} -> {adata.n_obs:,}")
print(f"Labeled: {n_labeled:,}  |  Unknown: {n_unknown:,}")
print(f"\nHierarchy summary:")
print(f"  L1 = Stromal (all cells)")
print(f"  L2 distribution:")
for lbl, n in adata.obs[OUT_L2].value_counts().items():
    print(f"    {lbl}: {n:,}")
print(f"  L3: {adata.obs[OUT_L3].nunique()} unique cluster prefixes")
print(f"\nNext steps:")
print(f"  1. Confirm TISSUE_KEY / LUNG_TRACHEA_VALUES match your data")
print(f"  2. Review scanvi_umap_overview.pdf -- L3 label vs prediction agreement")
print(f"  3. Check scanvi_label_summary.csv for high-uncertainty REVIEW clusters")



STEP 13: SAVE OUTPUT
Cells: 56,689 | HVGs: 4,000 | .raw: 36,789
[OK] /home/h2048/data/py/0308/stromal_reintegration_v1_3/stromal_reintegrated_scvi_scanvi_v1_3.h5ad

PIPELINE COMPLETE  (107.1 min)
Cells: 60,828 -> 56,689
Labeled: 40,007  |  Unknown: 16,682

Hierarchy summary:
  L1 = Stromal (all cells)
  L2 distribution:
    Endothelial: 34,362
    Fibroblast: 17,848
    Smooth_Muscle: 2,372
    Pericyte: 1,967
    Schwann: 140
  L3: 18 unique cluster prefixes

Next steps:
  1. Confirm TISSUE_KEY / LUNG_TRACHEA_VALUES match your data
  2. Review scanvi_umap_overview.pdf -- L3 label vs prediction agreement
  3. Check scanvi_label_summary.csv for high-uncertainty REVIEW clusters
